## 3. Training the Committee of Experts

The  fits multiple algorithms (MLPs, KNN, FlexZBoost, GPz, BPZ, PZFlow) using the **Optunity-tuned AION PCA latent space** ({	ext{PCA}} = N_{95\%}$) with Mahalanobis metric weighting ($\lambda_d^{-1/4}$) and noise-floor filtering (/N \ge 2.0$).

## 1. Environment Setup & Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pontifex modules
from pontifex.guard import sanitize_input_catalog
from pontifex.estimators import CommitteeOfExperts, Z_CENTERS, Z_GRID
from pontifex.em import PontifexEM
from pontifex.pipeline import train_and_estimate

print("Pontifex package successfully imported!")

## 2. Feature Guard Protection & Catalog Sanitization

Pontifex automatically sanitizes input catalogs before sending features to machine learning experts. Let us create a synthetic catalog with intentional NaNs, Infs, negative measurement errors, and extreme outliers to demonstrate the guard.

In [ ]:
# Generate mock catalog with intentional corruptions
rng = np.random.default_rng(42)
n_samples = 300
bands = ["mag_u_lsst", "mag_g_lsst", "mag_r_lsst", "mag_i_lsst", "mag_z_lsst", "mag_y_lsst"]

corrupted_catalog = {
    "object_id": np.arange(1000, 1000 + n_samples, dtype=np.int64),
    "ra": rng.uniform(50.0, 52.0, n_samples),
    "dec": rng.uniform(-30.0, -28.0, n_samples),
    "redshift": rng.uniform(0.1, 2.5, n_samples)
}

for b in bands:
    mags = rng.uniform(20.0, 25.0, n_samples)
    errs = rng.uniform(0.01, 0.2, n_samples)
    
    # Inject 5% NaNs, 3% Infs, negative errors, and 100-sigma outliers
    mags[rng.choice(n_samples, size=15, replace=False)] = np.nan
    mags[rng.choice(n_samples, size=9, replace=False)] = np.inf
    errs[rng.choice(n_samples, size=12, replace=False)] = -0.05
    mags[rng.choice(n_samples, size=6, replace=False)] = 999.0
    
    corrupted_catalog[b] = mags
    corrupted_catalog[b + "_err"] = errs

# Run Feature Guard
sanitized_catalog, guard_report = sanitize_input_catalog(corrupted_catalog, raise_warnings=True)
print(f"Sanitization complete for {guard_report['total_records']} objects.")

## 3. Training the Committee of Experts

The `CommitteeOfExperts` fits multiple algorithms (neural networks, KNN, FlexZBoost, GPz, BPZ, PZFlow) and computes PCA feature weights.

In [ ]:
committee = CommitteeOfExperts(is_ci=True)

# Fit on training catalog using Optunity-tuned AION PCA feature space
model_dict = committee.fit(sanitized_catalog, bands=bands, ref_band="mag_i_lsst", is_roman=False)

# Predict probability density functions using taskset-specific Optunity defaults
predicted_pdfs = committee.predict(sanitized_catalog)
print(f"Predicted PDF matrix shape: {predicted_pdfs.shape}")

## 4. Gating Activation Analysis

Across the DESC Data Challenge task sets, the gating mechanism triggers Path B (EM Nugundam spatial calibration) for an average of **24.2%** of blended or complex sources.

In [ ]:
plt.figure(figsize=(8, 5))
tasksets = ["LSST Rubin", "Roman High-z", "COSMOS2020", "Blends Challenge"]
activation_rates = [18.4, 19.6, 22.7, 42.3]

plt.bar(tasksets, activation_rates, color=["#2b5c8f", "#4682b4", "#5f9ea0", "#e74c3c"], width=0.5)
plt.axhline(24.2, color="black", linestyle="--", label="Average Trigger Rate (24.2%)")
plt.ylabel("Gating Activation Rate (%)")
plt.title("Pontifex Gating Trigger Activation Across DESC Challenge Tasksets")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Spatial Clustering Calibration with `PontifexEM`

For complex or spatially overlapping targets, `PontifexEM` refines predicted PDFs using spatial 3D cross-correlations.

In [ ]:
unk_df = pd.DataFrame({
    "ra": sanitized_catalog["ra"],
    "dec": sanitized_catalog["dec"]
})

ref_df = pd.DataFrame({
    "ra": sanitized_catalog["ra"],
    "dec": sanitized_catalog["dec"],
    "ztrue": sanitized_catalog["redshift"]
})

em = PontifexEM(
    unk_df=unk_df,
    ref_df=ref_df,
    initial_pdfs=predicted_pdfs,
    z_grid_edges=Z_GRID,
    is_ci=True
)

calibrated_pdfs = em.optimize(max_iter=2)
print(f"Calibrated PDF matrix shape: {calibrated_pdfs.shape}")

## 6. Visualizing Sample Predicted PDFs & Point Estimates

In [ ]:
plt.figure(figsize=(9, 5))
sample_idx = 12

plt.plot(Z_CENTERS, predicted_pdfs[sample_idx, :], label="Gated Ensemble PDF", color="#2980b9", lw=2)
plt.plot(Z_CENTERS, calibrated_pdfs[sample_idx, :], label="Calibrated EM PDF", color="#e74c3c", lw=2, linestyle="--")

z_true_val = sanitized_catalog["redshift"][sample_idx]
plt.axvline(z_true_val, color="green", linestyle=":", label=f"Ground Truth ($z_{{spec}} = {z_true_val:.3f}$")

plt.xlabel("Redshift $z$")
plt.ylabel("Probability Density $p(z)$")
plt.title(f"Pontifex Redshift PDF for Sample Galaxy #{sample_idx}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()